# Extensión del semillerío a k=20 — Experimento 9101

Este notebook agrega **15 semillas nuevas** al semillerío ya existente
(reusa las 5 predicciones guardadas en `./semillas/` por el 9101), y sube
el ensemble k=20 a Kaggle en varios cortes.

**Requisito:** el pipeline del notebook 9101 debe haber corrido hasta
`dfinal_train`, `param_final`, `campos_buenos` y `mfuture`. Si esa sesión
sigue viva, arrancá directo desde la celda 2. Si la perdiste, correr primero
todo el 9101 hasta la celda del `param_final` (sin re-hacer el loop de
semillerío ni el submit, ya están hechos).

In [ ]:
# --- Sanity check: los objetos del pipeline están en memoria? ---
require("data.table")
require("lightgbm")

obligatorios <- c("dfinal_train", "param_final", "campos_buenos", "mfuture", "dfuture")
faltantes <- obligatorios[!sapply(obligatorios, exists)]

if (length(faltantes) > 0) {
  stop("Faltan objetos en memoria: ", paste(faltantes, collapse = ", "),
       "\nCorré el notebook 9101 hasta la celda que arma param_final antes de continuar.")
}

cat("Todos los objetos necesarios están en memoria. OK para continuar.\n")
cat("Filas de dfinal_train:", dfinal_train$dim()[1], "\n")
cat("Filas de mfuture:     ", nrow(mfuture), "\n")
cat("num_iterations final: ", param_final$num_iterations, "\n")

In [ ]:
# --- Semillas nuevas a agregar (las primeras 15 del banco de 100 primos) ---

PARAM$semillerio$semillas_originales <- c(
  804043, 653561, 703903, 439693, 665857
)

PARAM$semillerio$semillas_nuevas <- c(
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973
)

PARAM$semillerio$semillas_k20 <- c(
  PARAM$semillerio$semillas_originales,
  PARAM$semillerio$semillas_nuevas
)

stopifnot(length(PARAM$semillerio$semillas_k20) == 20)
stopifnot(length(unique(PARAM$semillerio$semillas_k20)) == 20)

cat("Total de semillas en el ensemble k=20:",
    length(PARAM$semillerio$semillas_k20), "\n")

# chequeo cuántas semillas ya están en disco
dir.create("semillas", showWarnings = FALSE)
ya_entrenadas <- sapply(PARAM$semillerio$semillas_k20, function(s) {
  file.exists(paste0("semillas/prediccion_semilla_", s, ".txt"))
})

cat("Ya en disco: ", sum(ya_entrenadas), "\n")
cat("A entrenar:  ", sum(!ya_entrenadas), "\n")

In [ ]:
# --- Loop de entrenamiento de las semillas que faltan ---
# Idempotente: si el archivo ya existe en disco, salta esa semilla.

semillas_pendientes <- PARAM$semillerio$semillas_k20[!ya_entrenadas]

for (i in seq_along(semillas_pendientes)) {

  semilla <- semillas_pendientes[i]
  cat(format(Sys.time(), "%X"),
      " - Nueva semilla ", i, "/", length(semillas_pendientes),
      " = ", semilla, "\n", sep = "")

  param_semilla <- param_final
  param_semilla$seed <- semilla

  modelo_i <- lgb.train(
    data = dfinal_train,
    param = param_semilla,
    verbose = -100
  )

  prob_i <- predict(modelo_i, mfuture)

  tb_pred_i <- dfuture[, list(numero_de_cliente)]
  tb_pred_i[, prob := prob_i]
  fwrite(tb_pred_i,
    file = paste0("semillas/prediccion_semilla_", semilla, ".txt"),
    sep = "\t"
  )

  rm(modelo_i, prob_i, tb_pred_i)
  gc(full = TRUE, verbose = FALSE)
}

cat("\nEntrenamiento completado. Todas las semillas en disco.\n")

In [ ]:
# --- Cargar las 20 predicciones y armar el ensemble k=20 ---

tb_probs20 <- dfuture[, list(numero_de_cliente)]

for (semilla in PARAM$semillerio$semillas_k20) {
  tb_ind <- fread(paste0("semillas/prediccion_semilla_", semilla, ".txt"))
  # aseguro el mismo orden que numero_de_cliente
  tb_ind <- tb_ind[match(tb_probs20$numero_de_cliente, tb_ind$numero_de_cliente)]
  col_semilla <- paste0("prob_", semilla)
  tb_probs20[, (col_semilla) := tb_ind$prob]
}

cols_prob20 <- grep("^prob_", colnames(tb_probs20), value = TRUE)
stopifnot(length(cols_prob20) == 20)

tb_prediccion_k20 <- tb_probs20[, list(numero_de_cliente)]
tb_prediccion_k20[, prob := rowMeans(tb_probs20[, ..cols_prob20])]

fwrite(tb_prediccion_k20, file = "prediccion_k20.txt", sep = "\t")
fwrite(tb_probs20, file = "probs_por_semilla_k20.txt", sep = "\t")

cat("Ensemble k=20 armado y guardado.\n")

In [ ]:
# --- Análisis local: qué cambió al pasar de k=5 a k=20 ---

PARAM$kaggle$corte_referencia <- 2000  # el corte donde k=5 dio su máximo

# top 2000 del ensemble k=20
tb_prediccion_k20_sorted <- copy(tb_prediccion_k20)
setorder(tb_prediccion_k20_sorted, -prob)
top_k20 <- tb_prediccion_k20_sorted[1:PARAM$kaggle$corte_referencia, numero_de_cliente]

# top 2000 del ensemble k=5 (solo las 5 originales)
cols_prob5 <- paste0("prob_", PARAM$semillerio$semillas_originales)
tb_pred5 <- tb_probs20[, list(numero_de_cliente)]
tb_pred5[, prob := rowMeans(tb_probs20[, ..cols_prob5])]
setorder(tb_pred5, -prob)
top_k5 <- tb_pred5[1:PARAM$kaggle$corte_referencia, numero_de_cliente]

# comparación
en_ambos <- length(intersect(top_k20, top_k5))
cat("Clientes en el top-2000 tanto en k=5 como en k=20:", en_ambos, "/",
    PARAM$kaggle$corte_referencia,
    "(", round(en_ambos / PARAM$kaggle$corte_referencia, 3), ")\n")

# correlación media entre las 20 semillas
mat_probs20 <- as.matrix(tb_probs20[, ..cols_prob20])
cor_matrix20 <- cor(mat_probs20)
cor_media20 <- mean(cor_matrix20[upper.tri(cor_matrix20)])
cat("Correlación media entre 20 semillas:", round(cor_media20, 4), "\n")

# estabilidad del top-2000: en cuántas de las 20 semillas cada cliente cae en el top?
for (col in cols_prob20) {
  tb_probs20[, (paste0("in_top_", col)) :=
               numero_de_cliente %in% {
                 tb_tmp <- tb_probs20[, list(numero_de_cliente, p = get(col))]
                 setorder(tb_tmp, -p)
                 tb_tmp[1:PARAM$kaggle$corte_referencia, numero_de_cliente]
               }]
}
cols_in_top <- grep("^in_top_", colnames(tb_probs20), value = TRUE)
tb_probs20[, votos := rowSums(.SD), .SDcols = cols_in_top]

# distribución de votos: cuántos clientes fueron elegidos por 1, 2, ..., 20 semillas?
cat("\nDistribución de votos (cuántos clientes eligió cada N de semillas):\n")
print(tb_probs20[votos > 0, .N, by = votos][order(-votos)])

In [ ]:
# --- Submit del ensemble k=20 a Kaggle ---

PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

setorder(tb_prediccion_k20, -prob)
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion_k20[, Predicted := 0L]
  tb_prediccion_k20[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento,
                           "_ensemble_k20_", envios, ".csv")

  fwrite(tb_prediccion_k20[, list(numero_de_cliente, Predicted)],
         file = archivo_kaggle, sep = ",")

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'ensemble k=20 envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  cat(format(Sys.time(), "%X"), " - submit k=20 envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nSubmits del ensemble k=20 completados. Comparalos con los del k=5.\n")